# 03LIC_1071 PVLO — Alarm Episode Trajectory Analysis

**Goal of this notebook (Step 1 of a staged analysis).** Before we rebuild the similarity / control-action recommender, we need to *understand our alarm population*. Concretely, this notebook:

1. **Inventories all 539 PVLO alarm episodes** and tags each one with how the SME filtering pipeline treated it (which stage it was eliminated at, or whether it was retained).
2. **Builds an interpretable per-episode trajectory descriptor** for the target tag `03LIC_1071.PV` — a small set of shape features (how deep, how fast, how long, how oscillatory) that can be **compared across alarms**. This descriptor is the foundation of the similarity approach.
3. **Distinguishes "genuine dips" from "oscillatory grazes"** — the phenomenon you observed where the PV oscillates for hours and just touches the limit before bouncing back.
4. **Demonstrates a cross-alarm similarity metric** on the descriptor (is alarm A's 1071 trajectory moving similarly to alarm B's?).

**What this notebook deliberately does NOT do yet** (next steps, to agree on together):
- Root-cause attribution (feed-fluctuation vs compressor-pressure / `03PIC_1013`).
- Mapping trajectory severity → control-action magnitude (the OTS test requirement).

### Why this ordering
The OTS test the seniors want is: run the algorithm under 3 feed-variation scenarios of increasing severity and check that the **suggested action magnitude scales with severity** (and points the right direction). To make magnitude scale with severity, we first need a defensible, data-grounded notion of *severity* and *trajectory shape*. That is exactly what the descriptor below gives us.

> Data source: `RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/` — 539 per-episode folders (`*_pv_data.csv`, `*_events.csv`, `*_plot.html`) plus `filtering_log.json` (the SME funnel). Episode folder number == cluster_id within this run.


In [13]:
import os
import json
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

# ---- Paths ----
RUN_DIR = '/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219'
ALL_EPISODES_DIR = os.path.join(RUN_DIR, 'all_episodes')
FILTERING_LOG = os.path.join(RUN_DIR, 'filtering_log.json')
# Events-based alarm clustering — authoritative alarm window (cluster_start_time / cluster_end_time)
CLUSTER_WORKBOOK = os.path.join(RUN_DIR, '03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx')
OP_LIMITS_CSV = '/home/h604827/ControlActions/DATA/operating_limits.csv'
OUT_DIR = '/home/h604827/ControlActions/RESULTS/episode_trajectory_analysis'
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Constants ----
TARGET_TAG = '03LIC_1071'
TARGET_PV = '03LIC_1071.PV'
ALARM_THRESHOLD = 28.75      # PVLO low limit for 03LIC_1071
FEED_PV = '02FI_1000.PV'     # plant feed flow (main feed-fluctuation signal)
PRESSURE_PV = '03PIC_1013.PV'  # compressor pressure (second main cause)

# Operating limits (for normalising trajectories later)
op_limits_df = pd.read_csv(OP_LIMITS_CSV)
print('operating_limits.csv columns:', list(op_limits_df.columns))
op_limits_df.head()


operating_limits.csv columns: ['TAG_NAME', 'LOWER_LIMIT', 'UPPER_LIMIT', 'USER_MODIFIED', 'CONFIDENCE']


,TAG_NAME,LOWER_LIMIT,UPPER_LIMIT,USER_MODIFIED,CONFIDENCE
0,02FI_1000.PV,8.386506,8.630340,False,1.0
1,03FI_1141A.PV,72216.703130,72216.710940,False,NaN
2,03FI_1151.PV,249151.217100,285673.251700,False,1.0
3,03FIC_1085.OP,33.796284,42.461887,False,1.0
4,03FIC_1085.PV,247.113552,280.879444,False,1.0


## 1. Episode inventory + SME filter outcome

Each `episode_XXXX` folder corresponds to `cluster_id = XXXX`. The SME filtering pipeline (`filtering_log.json`) is a 4-stage funnel that took **539 raw episodes → 117 retained**. We attach the *outcome* of that funnel to every episode so we can later judge whether the SME criteria actually isolate the alarm population we care about.

| Stage | Criterion | What it removes | Eliminated |
|-------|-----------|-----------------|-----------:|
| 1 | duration ≥ 2 min | momentary single-sample grazes | 29 |
| 2 | FI1000 in band (±5%) | alarms where feed flow was *not* abnormal | 342 |
| 3 | quiet 4h look-back | alarms not preceded by a calm period (i.e. part of a messy run) | 45 |
| 4 | has OP/SP/MODE actions | alarms with no operator response to learn from | 6 |
| — | **retained** | candidate "clean, feed-driven, actionable" alarms | **117** |

Note Stage 2 is by far the biggest cut and encodes the SME assumption *"the alarm is feed-driven"*. We will keep **all 539** for the trajectory analysis and treat the SME label as just another column — so we can independently check whether the retained set really is more homogeneous.


In [14]:
# Load the SME filtering funnel and build a per-episode outcome map
with open(FILTERING_LOG) as f:
    flog = json.load(f)

elim = flog['eliminated_cluster_ids']
retained_ids = set(flog['retained_cluster_ids'])

# Map cluster_id -> filter outcome label
stage_label = {
    'stage_1': 'elim_S1_too_short',
    'stage_2': 'elim_S2_feed_in_band',
    'stage_3': 'elim_S3_not_quiet',
    'stage_4': 'elim_S4_no_actions',
}
outcome = {}
for skey, label in stage_label.items():
    for cid in elim[skey]:
        outcome[cid] = label
for cid in retained_ids:
    outcome[cid] = 'retained'

# Enumerate all episode folders present on disk
ep_folders = sorted(glob.glob(os.path.join(ALL_EPISODES_DIR, 'episode_*')))
inv_rows = []
for folder in ep_folders:
    cid = int(os.path.basename(folder).split('_')[1])
    pv_csv = os.path.join(folder, f'episode_{cid:04d}_pv_data.csv')
    ev_csv = os.path.join(folder, f'episode_{cid:04d}_events.csv')
    inv_rows.append({
        'cluster_id': cid,
        'folder': folder,
        'pv_csv': pv_csv if os.path.exists(pv_csv) else None,
        'events_csv': ev_csv if os.path.exists(ev_csv) else None,
        'filter_outcome': outcome.get(cid, 'unknown'),
        'sme_retained': cid in retained_ids,
    })

inventory = pd.DataFrame(inv_rows).sort_values('cluster_id').reset_index(drop=True)

# --- Authoritative alarm window: cluster_start_time / cluster_end_time ---
# We deliberately use the events-based cluster boundaries instead of the PV-file
# 'AlarmStatus' column, which is unreliable: 4 episodes have NO 'ON' samples at all,
# the onset differs by >5 min in ~200 episodes, and the trough pv_min shifts in ~23%.
ac = pd.read_excel(CLUSTER_WORKBOOK, sheet_name='alarm_clusters')
cluster_bounds_df = (
    ac.groupby('cluster_id')
      .agg(cluster_start=('cluster_start_time', 'first'),
           cluster_end=('cluster_end_time', 'first'),
           n_alarms=('cluster_total_alarms', 'first'),
           cluster_type=('cluster_type', 'first'))
      .reset_index()
)
cluster_bounds_df['cluster_start'] = pd.to_datetime(cluster_bounds_df['cluster_start'])
cluster_bounds_df['cluster_end'] = pd.to_datetime(cluster_bounds_df['cluster_end'])

inventory = inventory.merge(cluster_bounds_df, on='cluster_id', how='left')

# Fast lookup used by the geometry helpers: cluster_id -> (start, end)
cluster_bounds = {
    int(row.cluster_id): (row.cluster_start, row.cluster_end)
    for row in cluster_bounds_df.itertuples()
}

print(f'Episodes on disk:    {len(inventory)}')
print(f'With PV data:        {inventory["pv_csv"].notna().sum()}')
print(f'With events:         {inventory["events_csv"].notna().sum()}')
print(f'With cluster bounds: {inventory["cluster_start"].notna().sum()}')
print('\nFilter outcome distribution:')
print(inventory['filter_outcome'].value_counts().to_string())
inventory.head()


Episodes on disk:    539
With PV data:        539
With events:         525
With cluster bounds: 539

Filter outcome distribution:
filter_outcome
elim_S2_feed_in_band    342
retained                117
elim_S3_not_quiet        45
elim_S1_too_short        29
elim_S4_no_actions        6


,cluster_id,folder,pv_csv,events_csv,filter_outcome,sme_retained,cluster_start,cluster_end,n_alarms,cluster_type
0,1,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,elim_S2_feed_in_band,False,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,1,Medium situation (<2h)
1,2,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,elim_S2_feed_in_band,False,2022-01-07 09:55:16.253,2022-01-07 10:00:27.504,1,Small cluster
2,3,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,elim_S2_feed_in_band,False,2022-01-07 13:33:25.702,2022-01-07 13:36:35.204,1,Isolated brief alarm
3,4,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,elim_S2_feed_in_band,False,2022-01-07 14:17:11.555,2022-01-07 14:19:30.553,1,Isolated brief alarm
4,5,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,/home/h604827/ControlActions/RESULTS/03LIC_107...,elim_S2_feed_in_band,False,2022-01-07 14:54:23.554,2022-01-07 14:58:16.555,1,Isolated brief alarm


## 2. Episode loader + alarm-window geometry

Inspecting a few episodes shows a consistent layout (matches the run config `-240 / +60 min`):

```
[ 240 min pre-alarm context ]  [ alarm window (variable) ]  [ 60 min after ]
```

**Alarm window source — cluster boundaries, not `AlarmStatus`.** The per-minute PV file carries an `AlarmStatus` column, but it is unreliable here: **4 episodes have no `ON` samples at all**, the onset differs by >5 min from the cluster start in **~200 episodes**, and the trough `pv_min` shifts in **~23%** of episodes. Instead we use the **events-based cluster boundaries** (`cluster_start_time` / `cluster_end_time` from the `alarm_clusters` sheet), which come from the actual DCS alarm records and are sub-second precise. For every episode we recover:
- **onset** = `cluster_start_time`, **end** = `cluster_end_time`
- **trough** = time of minimum `03LIC_1071.PV` in `[onset−5min, end+5min]`
- **baseline window** = the pre-alarm context (used for volatility / oscillation)
- **approach / recovery windows** = around the trough (used for descent / ascent rates)

A first quick scan already reveals the phenomena you described:
- ep 0001: PV crashes to **0.07** (a trip/shutdown, not a control problem) → must be flagged & excluded.
- ep 0290: a single brief alarm — PV just grazes 27.99 then bounces back.
- ep 0064: a sustained dip to 28.27 over a ~50-min cluster → "genuine dip".


In [15]:
def load_episode_pv(cid):
    """Load one episode's PV dataframe, indexed by TimeStamp (minute grid)."""
    row = inventory.loc[inventory['cluster_id'] == cid].iloc[0]
    df = pd.read_csv(row['pv_csv'], parse_dates=['TimeStamp'])
    df = df.sort_values('TimeStamp').set_index('TimeStamp')
    return df


def alarm_geometry(df, cid):
    """Recover onset / end / trough timestamps and the key sub-windows.

    The alarm window comes from the events-based cluster boundaries
    (cluster_start_time / cluster_end_time), NOT the unreliable 'AlarmStatus'
    column. Returns a dict (or None if bounds / target column are missing).
    """
    if TARGET_PV not in df.columns:
        return None
    bounds = cluster_bounds.get(int(cid))
    if bounds is None or pd.isna(bounds[0]) or pd.isna(bounds[1]):
        return None
    onset, end = bounds
    pv = df[TARGET_PV]

    # Trough = min PV in a small pad around the cluster span
    trough_win = pv.loc[onset - pd.Timedelta(minutes=5): end + pd.Timedelta(minutes=5)]
    if trough_win.dropna().empty:
        return None
    trough_t = trough_win.idxmin()

    n_alarms = inventory.loc[inventory['cluster_id'] == cid, 'n_alarms']
    n_alarms = int(n_alarms.iloc[0]) if len(n_alarms) and pd.notna(n_alarms.iloc[0]) else 1

    return {
        'onset': onset,
        'end': end,
        'trough_t': trough_t,
        'pv_min': float(trough_win.min()),
        'n_alarms': n_alarms,                                        # sub-alarms merged into the cluster
        'baseline': pv.loc[:onset],                                  # pre-alarm context
        'approach': pv.loc[onset - pd.Timedelta(minutes=30): trough_t],
        'recovery': pv.loc[trough_t: trough_t + pd.Timedelta(minutes=30)],
        'full': pv,
    }


# Smoke-test on the three example episodes
for cid in [64, 290, 1]:
    g = alarm_geometry(load_episode_pv(cid), cid)
    print(f'ep {cid:>3}: onset={g["onset"]}  end={g["end"]}  '
          f'pv_min={g["pv_min"]:.2f}  n_alarms={g["n_alarms"]}  '
          f'baseline_std={g["baseline"].std():.2f}')


ep  64: onset=2022-04-14 16:07:34.456000  end=2022-04-14 16:58:01.954000  pv_min=28.27  n_alarms=1  baseline_std=1.31
ep 290: onset=2024-04-18 22:28:26.855000  end=2024-04-18 22:31:28.854000  pv_min=27.99  n_alarms=1  baseline_std=1.32
ep   1: onset=2022-01-05 08:53:41.853000  end=2022-01-05 09:33:33.105000  pv_min=0.07  n_alarms=1  baseline_std=4.60


## 3. Per-episode trajectory descriptor

This is the core of the notebook: a compact, **interpretable** set of features describing the shape of the `03LIC_1071.PV` trajectory around each alarm. The features are grouped by the question they answer:

| Group | Feature | Question it answers |
|-------|---------|---------------------|
| **Depth** | `depth_below_limit`, `pv_min`, `pv_at_onset` | *How severe* is the breach? |
| **Descent** | `approach_slope_30`, `max_descent_rate` | *How fast* is it falling into the alarm? |
| **Recovery** | `recovery_slope_30` | *How fast* does it come back? |
| **Duration** | `alarm_duration_min` | *How long* does it stay in alarm? |
| **Oscillation** | `baseline_std`, `n_troughs_near_limit`, `trough_prominence_norm`, `frac_below_limit` | Is this a *one-off dip* or *ongoing oscillation grazing the limit*? |
| **Context** | `feed_at_onset`, `feed_slope_30`, `pressure_at_onset`, `pressure_slope_30` | (light touch) feed vs compressor-pressure fingerprint, for the later RC step |
| **Flags** | `is_trip` | plant shutdown (PV → 0), to be excluded |

**Design choices worth flagging for review:**
- *Descent / recovery slopes* are robust linear fits (`np.polyfit`) over a 30-min window, in **%/min**. These are the quantities the OTS test cares about — they translate directly into "how urgent / how big an action".
- *Oscillation* is captured two ways: `baseline_std` (pre-alarm volatility) and `n_troughs_near_limit` (count of distinct dips that reach within `graze_band` of the limit, via `scipy.signal.find_peaks`). A high count + short duration + low prominence = your "oscillatory graze".
- `trough_prominence_norm` = how far the alarm trough stands out below the local baseline, **in units of baseline σ**. Genuine dips stand out (>~2σ); grazes barely do.


In [16]:
from scipy.signal import find_peaks

GRAZE_BAND = 1.0   # PV within this many units of the limit counts as "near the limit"
TRIP_PV = 5.0      # PV below this is treated as a trip / shutdown, not a control excursion


def _slope_per_min(series):
    """Robust linear slope in units/min over a time-indexed series (NaN-safe)."""
    s = series.dropna()
    if len(s) < 3:
        return np.nan
    t = (s.index - s.index[0]).total_seconds().values / 60.0
    if t[-1] == t[0]:
        return np.nan
    return float(np.polyfit(t, s.values, 1)[0])


def _max_descent_rate(series, win=5):
    """Most negative rolling `win`-min slope inside the approach window (units/min)."""
    s = series.dropna()
    if len(s) < win + 1:
        return _slope_per_min(s)
    rates = [_slope_per_min(s.iloc[i:i + win + 1]) for i in range(len(s) - win)]
    rates = [r for r in rates if pd.notna(r)]
    return float(min(rates)) if rates else np.nan


def _ctx(df, col, onset):
    """Value at onset and 30-min approach slope for a context tag (feed / pressure)."""
    if col not in df.columns:
        return np.nan, np.nan
    s = df[col]
    val = s.loc[:onset].iloc[-1] if len(s.loc[:onset]) else np.nan
    sl = _slope_per_min(s.loc[onset - pd.Timedelta(minutes=30):onset])
    return (float(val) if pd.notna(val) else np.nan), sl


def describe_episode(cid):
    """Compute the interpretable trajectory descriptor for one episode."""
    df = load_episode_pv(cid)
    g = alarm_geometry(df, cid)
    if g is None:
        return None

    pv_full = g['full'].dropna()
    baseline = g['baseline'].dropna()
    base_std = float(baseline.std()) if len(baseline) > 2 else np.nan
    base_med = float(baseline.median()) if len(baseline) else np.nan

    # --- trip detection: PV crashes toward zero ---
    sustained_low = (pv_full < TRIP_PV).sum()
    is_trip = bool(g['pv_min'] < TRIP_PV and sustained_low >= 3)

    # --- oscillation: count distinct troughs that reach near the limit ---
    # find_peaks on the inverted signal; prominence in units of baseline noise
    inv = -pv_full.values
    prom = max(0.3, 0.5 * base_std) if pd.notna(base_std) else 0.3
    peak_idx, peak_props = find_peaks(inv, prominence=prom)
    trough_vals = pv_full.values[peak_idx]
    n_troughs_total = int(len(peak_idx))
    n_troughs_near_limit = int(np.sum(trough_vals <= ALARM_THRESHOLD + GRAZE_BAND))

    # prominence of the alarm trough itself, normalised by baseline noise
    if n_troughs_total > 0:
        # match the trough nearest to g['trough_t']
        trough_pos = pv_full.index.get_indexer([g['trough_t']], method='nearest')[0]
        nearest = peak_idx[np.argmin(np.abs(peak_idx - trough_pos))]
        alarm_prom = float(peak_props['prominences'][np.argmin(np.abs(peak_idx - trough_pos))])
    else:
        alarm_prom = float(base_med - g['pv_min']) if pd.notna(base_med) else np.nan
    trough_prom_norm = (alarm_prom / base_std) if (pd.notna(base_std) and base_std > 0) else np.nan

    frac_below_limit = float((pv_full < ALARM_THRESHOLD).mean())

    feed_at_onset, feed_slope_30 = _ctx(df, FEED_PV, g['onset'])
    pres_at_onset, pres_slope_30 = _ctx(df, PRESSURE_PV, g['onset'])

    return {
        'cluster_id': cid,
        'onset': g['onset'], 'end': g['end'], 'trough_t': g['trough_t'],
        # depth
        'pv_at_onset': float(g['baseline'].iloc[-1]) if len(g['baseline']) else np.nan,
        'pv_min': g['pv_min'],
        'depth_below_limit': max(0.0, ALARM_THRESHOLD - g['pv_min']),
        # descent / recovery
        'approach_slope_30': _slope_per_min(g['approach']),
        'max_descent_rate': _max_descent_rate(g['approach']),
        'recovery_slope_30': _slope_per_min(g['recovery']),
        # duration
        'alarm_duration_min': (g['end'] - g['onset']).total_seconds() / 60.0,
        'n_alarms': g['n_alarms'],
        # oscillation
        'baseline_std': base_std,
        'baseline_median': base_med,
        'n_troughs_total': n_troughs_total,
        'n_troughs_near_limit': n_troughs_near_limit,
        'trough_prominence': alarm_prom,
        'trough_prominence_norm': trough_prom_norm,
        'frac_below_limit': frac_below_limit,
        # context (feed / pressure)
        'feed_at_onset': feed_at_onset, 'feed_slope_30': feed_slope_30,
        'pressure_at_onset': pres_at_onset, 'pressure_slope_30': pres_slope_30,
        # flags
        'is_trip': is_trip,
    }


# Smoke-test on the three archetype episodes
test = pd.DataFrame([describe_episode(c) for c in [64, 290, 1]])
test[['cluster_id', 'pv_min', 'depth_below_limit', 'approach_slope_30',
      'recovery_slope_30', 'alarm_duration_min', 'baseline_std',
      'n_troughs_near_limit', 'trough_prominence_norm', 'frac_below_limit', 'is_trip']]


,cluster_id,pv_min,depth_below_limit,approach_slope_30,recovery_slope_30,alarm_duration_min,baseline_std,n_troughs_near_limit,trough_prominence_norm,frac_below_limit,is_trip
0,64,28.273735,0.476265,-0.016094,0.014824,50.458300,1.307248,7,3.161353,0.008547,False
1,290,27.990063,0.759937,-0.068895,0.015669,3.033317,1.316721,1,7.561714,0.006601,False
2,1,0.072141,28.677859,-0.906903,0.853986,39.854200,4.601275,3,8.633223,0.111765,True


In [17]:
# Compute the descriptor for all 539 episodes
records = []
for cid in tqdm(inventory['cluster_id'], desc='Describing episodes'):
    try:
        d = describe_episode(cid)
        if d is not None:
            records.append(d)
    except Exception as e:
        print(f'  ep {cid} failed: {e}')

desc = pd.DataFrame(records)
desc = desc.merge(inventory[['cluster_id', 'filter_outcome', 'sme_retained']], on='cluster_id')
desc['year'] = pd.to_datetime(desc['onset']).dt.year

print(f'Descriptor built for {len(desc)} episodes')
print(f'Trips flagged: {desc["is_trip"].sum()}')
print(f'\nFilter outcome vs trip flag:')
print(pd.crosstab(desc['filter_outcome'], desc['is_trip']))

# Save for downstream use
desc.to_csv(os.path.join(OUT_DIR, 'episode_trajectory_descriptors.csv'), index=False)
print(f'\nSaved -> {OUT_DIR}/episode_trajectory_descriptors.csv')


Describing episodes: 100%|██████████| 539/539 [00:07<00:00, 74.57it/s]

Descriptor built for 539 episodes
Trips flagged: 20

Filter outcome vs trip flag:
is_trip               False  True 
filter_outcome                    
elim_S1_too_short        29      0
elim_S2_feed_in_band    329     13
elim_S3_not_quiet        40      5
elim_S4_no_actions        6      0
retained                115      2

Saved -> /home/h604827/ControlActions/RESULTS/episode_trajectory_analysis/episode_trajectory_descriptors.csv


### 3.1 Which slice of the −240/+60 signal each feature actually uses

You're right that each episode file spans roughly **−240 → +60 min**. But the descriptor does **not** compare that whole curve point-by-point. Each feature is computed over a *specific sub-window* chosen to answer one question:
- **depth / slopes** look only *around the alarm* — the trough and ±30 min of it (these are what the OTS test cares about: how deep, how fast).
- **volatility** uses the pre-alarm context.
- Only **two** features (`n_troughs_near_limit`, `frac_below_limit`) span the whole −240/+60 window.

The cell below prints the exact feature→window mapping. It then does an **honesty check** on those two full-window features: for the `oscillatory` episodes, it reports whether the near-limit troughs actually sit *around the alarm* or back in the *run-up* — so you can decide whether oscillation should be measured only in the alarm-local window.


In [26]:
# 3.1  Which slice of the -240/+60 window feeds each feature  + an honesty check
from scipy.signal import find_peaks

feature_windows = pd.DataFrame([
    ('pv_at_onset',                  'single value at onset (= cluster_start)'),
    ('pv_min, depth_below_limit',    'minimum in [onset-5min, end+5min]  (the trough)'),
    ('approach_slope_30',            '[onset-30min  ->  trough]'),
    ('max_descent_rate',             'steepest rolling 5-min slope inside [onset-30min -> trough]'),
    ('recovery_slope_30',            '[trough  ->  trough+30min]'),
    ('alarm_duration_min',           'cluster_end - cluster_start'),
    ('baseline_std, baseline_median','pre-alarm context [file start -> onset]  (up to 240 min)'),
    ('n_troughs_near_limit',         'WHOLE episode window  [-240min, +60min]'),
    ('frac_below_limit',             'WHOLE episode window  [-240min, +60min]'),
    ('feed/pressure at_onset+slope', 'value at onset, and [onset-30min -> onset] slope'),
], columns=['feature', 'window_used'])
print('Window each descriptor feature is computed over:\n')
print(feature_windows.to_string(index=False))
print('\n=> NO feature compares the raw -240/+60 curve point-by-point.')
print('   Most features look only AROUND the alarm (onset/trough +-30 min).')
print('   Only n_troughs_near_limit and frac_below_limit span the whole window.\n')

# Honesty check: for those two full-window features, WHERE do the near-limit troughs sit?
# (run-up before the alarm vs around the alarm). This tells us whether an "oscillatory"
# label reflects the alarm itself or just a noisy pre-context.
phase = []
for cid in desc.loc[~desc['is_trip'], 'cluster_id']:
    g = alarm_geometry(load_episode_pv(cid), cid)
    if g is None:
        continue
    pv = g['full'].dropna()
    bstd = g['baseline'].dropna().std()
    prom = max(0.3, 0.5 * bstd) if pd.notna(bstd) else 0.3
    pk, _ = find_peaks(-pv.values, prominence=prom)
    if len(pk) == 0:
        phase.append((cid, 0, 0, 0)); continue
    tt = pv.index[pk]; tv = pv.values[pk]
    tt = tt[tv <= ALARM_THRESHOLD + GRAZE_BAND]                      # near-limit troughs only
    o5 = g['onset'] - pd.Timedelta(minutes=5); e5 = g['end'] + pd.Timedelta(minutes=5)
    pre = int((tt < o5).sum())
    al = int(((tt >= o5) & (tt <= e5)).sum())
    post = int((tt > e5).sum())
    phase.append((cid, pre, al, post))

phase = pd.DataFrame(phase, columns=['cluster_id', 'troughs_pre', 'troughs_alarm', 'troughs_post'])
phase = phase.merge(desc[['cluster_id', 'character']], on='cluster_id')
osc_only = phase[phase['character'] == 'oscillatory']
print(f"Oscillatory episodes (n={len(osc_only)}) — median near-limit troughs by phase:")
print(f"  pre-context: {osc_only['troughs_pre'].median():.0f}   "
      f"alarm-local: {osc_only['troughs_alarm'].median():.0f}   "
      f"post: {osc_only['troughs_post'].median():.0f}")
share_pre = (osc_only['troughs_pre'] > (osc_only['troughs_alarm'] + osc_only['troughs_post'])).mean()
print(f"  {share_pre*100:.0f}% of 'oscillatory' episodes have MORE near-limit troughs in the run-up "
      f"than around the alarm itself.")
print("  -> if you'd rather measure oscillation only in the alarm-local window, that's a 1-line change.")


Window each descriptor feature is computed over:

                      feature                                                 window_used
                  pv_at_onset                     single value at onset (= cluster_start)
    pv_min, depth_below_limit             minimum in [onset-5min, end+5min]  (the trough)
            approach_slope_30                                   [onset-30min  ->  trough]
             max_descent_rate steepest rolling 5-min slope inside [onset-30min -> trough]
            recovery_slope_30                                  [trough  ->  trough+30min]
           alarm_duration_min                                 cluster_end - cluster_start
baseline_std, baseline_median    pre-alarm context [file start -> onset]  (up to 240 min)
         n_troughs_near_limit                     WHOLE episode window  [-240min, +60min]
             frac_below_limit                     WHOLE episode window  [-240min, +60min]
 feed/pressure at_onset+slope            value at 

## 4. What does the alarm population actually look like?

Before classifying, let's look at the distributions of the key shape features (trips excluded). This tells us where natural cut-points are, instead of hard-coding arbitrary thresholds.


In [18]:
# Distribution summary of key shape features (non-trip episodes)
nontrip = desc[~desc['is_trip']].copy()
key_feats = ['depth_below_limit', 'approach_slope_30', 'max_descent_rate',
             'recovery_slope_30', 'alarm_duration_min', 'baseline_std',
             'n_troughs_near_limit', 'trough_prominence_norm', 'frac_below_limit']

pct = nontrip[key_feats].describe(percentiles=[.1, .25, .5, .75, .9]).T
pct = pct[['min', '10%', '25%', '50%', '75%', '90%', 'max', 'mean']].round(3)
print(f'Non-trip episodes: {len(nontrip)}\n')
print(pct.to_string())


Non-trip episodes: 519

                           min    10%    25%    50%     75%     90%      max    mean
depth_below_limit        0.000  0.028  0.586  1.944   6.631  13.895   27.833   4.506
approach_slope_30       -2.585 -0.366 -0.226 -0.115  -0.046  -0.002    0.424  -0.172
max_descent_rate       -20.963 -6.966 -4.694 -3.089  -1.943  -1.165   -0.001  -3.608
recovery_slope_30       -0.580  0.004  0.092  0.184   0.319   0.477    3.444   0.230
alarm_duration_min       0.300  2.698  4.123  7.033  28.483  67.270  495.917  26.655
baseline_std             0.034  1.551  2.346  3.385   4.750   7.216   33.824   4.086
n_troughs_near_limit     0.000  1.000  1.000  3.000   5.000   8.200   59.000   3.944
trough_prominence_norm   0.638  3.058  4.130  5.743   7.640  11.469  618.879   7.886
frac_below_limit         0.000  0.003  0.010  0.025   0.060   0.105    0.429   0.045


### 4.1 A tunable archetype classifier

The distributions confirm the population is **not homogeneous**. Two axes matter:

- **Character** (the *shape*): trip / graze / oscillatory / dip
- **Severity** (the *depth*): how far below the limit → this is what should drive action magnitude

I encode *Character* as an ordered rule set (first match wins). **All thresholds are named constants** at the top of the cell — these are deliberately conservative defaults derived from the percentiles above, and are meant to be tuned together:

1. **trip** — `is_trip` (PV → 0, plant shutdown)
2. **oscillatory** — dips near the limit ≥ `OSC_TROUGHS` times but is below the limit < `OSC_FRAC` of the time → *repeatedly grazes without staying down* (your "oscillating for hours" case)
3. **graze** — very short (`≤ GRAZE_DUR_MIN`) and shallow (`≤ GRAZE_DEPTH`) → momentary touch
4. **sustained_dip** — stays below the limit a meaningful fraction (`≥ SUSTAINED_FRAC`) or for `≥ SUSTAINED_DUR_MIN` → a clear, prolonged excursion
5. **transient_dip** — everything else (a clean one-off dip that recovers fairly quickly)

*Severity* is a separate shallow/moderate/deep tier on `depth_below_limit`, so e.g. a "deep sustained_dip" and a "shallow graze" are both expressible.


In [25]:
# --- Tunable thresholds (derived from the percentile table above) ---
OSC_TROUGHS      = 5     # >= this many near-limit troughs -> repeated grazing
OSC_FRAC         = 0.10  # ...but below limit < this fraction of the window
GRAZE_DUR_MIN    = 5     # <= this alarm duration (min) = momentary
GRAZE_DEPTH      = 1.0   # <= this depth below limit  = barely breaches
SUSTAINED_FRAC   = 0.10  # >= this fraction below limit = prolonged
SUSTAINED_DUR_MIN = 60   # ...or >= this alarm duration (min)

# Severity tiers on depth_below_limit (from the percentiles)
DEPTH_SHALLOW = 1.0      # ~25th pct
DEPTH_DEEP    = 8.5      # ~75th pct


def classify_character(r):
    if r['is_trip']:
        return 'trip'
    if (r['n_troughs_near_limit'] >= OSC_TROUGHS) and (r['frac_below_limit'] < OSC_FRAC):
        return 'oscillatory'
    if (r['alarm_duration_min'] <= GRAZE_DUR_MIN) and (r['depth_below_limit'] <= GRAZE_DEPTH):
        return 'graze'
    if (r['frac_below_limit'] >= SUSTAINED_FRAC) or (r['alarm_duration_min'] >= SUSTAINED_DUR_MIN):
        return 'sustained_dip'
    return 'transient_dip'


def severity_tier(depth):
    if depth <= DEPTH_SHALLOW:
        return 'shallow'
    if depth >= DEPTH_DEEP:
        return 'deep'
    return 'moderate'


desc['character'] = desc.apply(classify_character, axis=1)
desc['severity'] = desc['depth_below_limit'].apply(severity_tier)

char_order = ['trip', 'graze', 'oscillatory', 'transient_dip', 'sustained_dip']
print(f'Character distribution (all {len(desc)} episodes):')
print(desc['character'].value_counts().reindex(char_order).to_string())

print('\nCharacter x Severity (non-trip):')
print(pd.crosstab(desc.loc[desc.character != "trip", 'character'],
                  desc.loc[desc.character != "trip", 'severity'])
        .reindex(index=[c for c in char_order if c != 'trip'],
                 columns=['shallow', 'moderate', 'deep']))

# Re-save with labels
desc.to_csv(os.path.join(OUT_DIR, 'episode_trajectory_descriptors.csv'), index=False)


Character distribution (all 539 episodes):
character
trip              20
graze             97
oscillatory      114
transient_dip    248
sustained_dip     60

Character x Severity (non-trip):
severity       shallow  moderate  deep
character                             
graze               97         0     0
oscillatory         26        52    36
transient_dip       50       168    30
sustained_dip        4        21    35


### 4.2 Visual validation — do the archetypes look right?

The only honest test of the labels is to **overlay the actual trajectories** and see whether each archetype has a recognisable shape. Below, every episode in an archetype is time-aligned at its **trough (t = 0)** and plotted as raw `03LIC_1071.PV`; the red dashed line is the alarm limit (28.75). A thick line shows the per-archetype median shape.


In [20]:
def aligned_pv(cid, lo=-120, hi=90):
    """Return PV (raw) on a minute axis centred at the trough (t=0)."""
    df = load_episode_pv(cid)
    g = alarm_geometry(df, cid)
    if g is None:
        return None
    pv = g['full']
    rel = (pv.index - g['trough_t']).total_seconds() / 60.0
    s = pd.Series(pv.values, index=np.round(rel).astype(int))
    s = s[(s.index >= lo) & (s.index <= hi)]
    s = s[~s.index.duplicated(keep='first')]
    return s


archetypes_to_plot = ['graze', 'oscillatory', 'transient_dip', 'sustained_dip', 'trip']
fig = make_subplots(rows=1, cols=5, shared_yaxes=True,
                    subplot_titles=[f'{a}<br>(n={int((desc.character==a).sum())})'
                                    for a in archetypes_to_plot])

rng = np.random.default_rng(7)
for col, arch in enumerate(archetypes_to_plot, start=1):
    cids = desc.loc[desc.character == arch, 'cluster_id'].tolist()
    sample = rng.choice(cids, size=min(30, len(cids)), replace=False)
    aligned = []
    for cid in sample:
        s = aligned_pv(cid)
        if s is None or s.empty:
            continue
        aligned.append(s)
        fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines',
                                 line=dict(color='lightgray', width=0.7),
                                 showlegend=False, hoverinfo='skip'), row=1, col=col)
    # median shape across the sample
    if aligned:
        med = pd.concat(aligned, axis=1).median(axis=1).sort_index()
        fig.add_trace(go.Scatter(x=med.index, y=med.values, mode='lines',
                                 line=dict(color='black', width=2.5),
                                 showlegend=False), row=1, col=col)
    fig.add_hline(y=ALARM_THRESHOLD, line_dash='dash', line_color='red', row=1, col=col)
    fig.add_vline(x=0, line_dash='dot', line_color='blue', opacity=0.4, row=1, col=col)

fig.update_yaxes(range=[0, 55], title_text='03LIC_1071.PV', row=1, col=1)
fig.update_xaxes(title_text='min from trough')
fig.update_layout(height=420, width=1400,
                  title_text='Archetype trajectories (aligned at trough; black = median shape)')
fig.show()
fig.write_html(os.path.join(OUT_DIR, 'archetype_trajectories.html'))
print(f'Saved -> {OUT_DIR}/archetype_trajectories.html')


Saved -> /home/h604827/ControlActions/RESULTS/episode_trajectory_analysis/archetype_trajectories.html


### 4.3 Should we trust the SME filtering? (funnel vs archetype)

You asked whether to accept the SME filtering and shrink the dataset, or not. Let's cross-tabulate the SME funnel outcome against our data-driven archetypes to see *what kind of episodes each stage throws away*.


In [21]:
# Archetype composition of each SME funnel outcome
ct = pd.crosstab(desc['filter_outcome'], desc['character'])
ct = ct.reindex(columns=char_order, fill_value=0)
ct = ct.reindex(['retained', 'elim_S1_too_short', 'elim_S2_feed_in_band',
                 'elim_S3_not_quiet', 'elim_S4_no_actions'])
print('Archetype composition by SME funnel outcome:\n')
print(ct.to_string())

print('\nRow-normalised (% of each funnel group):')
print((ct.div(ct.sum(axis=1), axis=0) * 100).round(1).to_string())

# How much of each archetype survives SME filtering?
print('\nRetention rate by archetype (% of that archetype kept by SME):')
surv = desc.groupby('character')['sme_retained'].mean().reindex(char_order) * 100
print(surv.round(1).to_string())


Archetype composition by SME funnel outcome:

character             trip  graze  oscillatory  transient_dip  sustained_dip
filter_outcome                                                              
retained                 2     32           16             63              4
elim_S1_too_short        0     21            3              5              0
elim_S2_feed_in_band    13     34           78            163             54
elim_S3_not_quiet        5      6           17             15              2
elim_S4_no_actions       0      4            0              2              0

Row-normalised (% of each funnel group):
character             trip  graze  oscillatory  transient_dip  sustained_dip
filter_outcome                                                              
retained               1.7   27.4         13.7           53.8            3.4
elim_S1_too_short      0.0   72.4         10.3           17.2            0.0
elim_S2_feed_in_band   3.8    9.9         22.8           47.7    

## 5. A cross-alarm trajectory similarity metric

The original ask: *given a new alarm, find historically similar 1071 trajectories so we can reuse the control actions that worked.* The descriptor above gives us a clean way to do that. I demonstrate **two complementary notions of "similar"**:

1. **Descriptor-space similarity** (fast, interpretable): represent each alarm as a standardised feature vector `[depth, descent slope, recovery slope, duration, volatility, #troughs, frac-below]`, then use Euclidean distance. This is the metric that will plug into the recommender.
2. **Shape similarity** (literal): correlation of the trough-aligned PV curves — does alarm A's PV *move* like alarm B's.

Below, for a couple of query alarms, I pull the 5 nearest neighbours in descriptor space and overlay their actual PV trajectories. If the metric is meaningful, the neighbours should visually resemble the query.


In [22]:
# Build a standardised descriptor matrix over non-trip episodes
sim_feats = ['depth_below_limit', 'approach_slope_30', 'recovery_slope_30',
             'alarm_duration_min', 'baseline_std', 'n_troughs_near_limit',
             'frac_below_limit']

D = desc[desc['character'] != 'trip'].copy().reset_index(drop=True)
X = D[sim_feats].astype(float)
# robust standardisation (median / IQR) so outliers don't dominate
med = X.median()
iqr = (X.quantile(0.75) - X.quantile(0.25)).replace(0, 1.0)
Xz = (X - med) / iqr
Xz = Xz.fillna(0).values


def neighbors(query_cid, k=5):
    qi = D.index[D['cluster_id'] == query_cid]
    if len(qi) == 0:
        return None
    qi = qi[0]
    dist = np.sqrt(((Xz - Xz[qi]) ** 2).sum(axis=1))
    order = np.argsort(dist)
    order = [i for i in order if i != qi][:k]
    out = D.iloc[order][['cluster_id', 'character', 'severity',
                         'depth_below_limit', 'alarm_duration_min',
                         'approach_slope_30', 'n_troughs_near_limit']].copy()
    out.insert(1, 'distance', dist[order].round(3))
    return out


# Pick one query per main archetype and show neighbours + overlay
query_cids = []
for arch in ['graze', 'oscillatory', 'transient_dip', 'sustained_dip']:
    sub = D[D['character'] == arch]
    if len(sub):
        # pick the median-depth example as a representative query
        q = sub.iloc[(sub['depth_below_limit'] - sub['depth_below_limit'].median()).abs().argsort().iloc[0]]
        query_cids.append(int(q['cluster_id']))

for qc in query_cids:
    qrow = D[D.cluster_id == qc].iloc[0]
    print(f'\n=== Query ep {qc}  [{qrow["character"]}, {qrow["severity"]}, '
          f'depth={qrow["depth_below_limit"]:.1f}, dur={qrow["alarm_duration_min"]:.0f}min] ===')
    print(neighbors(qc, k=5).to_string(index=False))



=== Query ep 377  [graze, shallow, depth=0.2, dur=2min] ===
 cluster_id  distance character severity  depth_below_limit  alarm_duration_min  approach_slope_30  n_troughs_near_limit
        415     0.385     graze  shallow           0.656212            2.566667          -0.169300                     1
        521     0.425     graze  shallow           0.000000            0.314050          -0.162212                     0
        515     0.493     graze  shallow           0.000000            0.735733          -0.167293                     0
        430     0.533     graze  shallow           0.137560            2.604200          -0.166856                     1
        335     0.577     graze  shallow           0.399704            2.604133          -0.218621                     1

=== Query ep 314  [oscillatory, moderate, depth=3.9, dur=46min] ===
 cluster_id  distance     character severity  depth_below_limit  alarm_duration_min  approach_slope_30  n_troughs_near_limit
        187     1.6

In [23]:
# Visual proof: overlay each query (bold red) with its 5 nearest neighbours (gray)
fig = make_subplots(rows=1, cols=len(query_cids), shared_yaxes=True,
                    subplot_titles=[f'query ep {qc} ({desc.loc[desc.cluster_id==qc,"character"].iloc[0]})'
                                    for qc in query_cids])

for col, qc in enumerate(query_cids, start=1):
    nbrs = neighbors(qc, k=5)
    for ncid in nbrs['cluster_id']:
        s = aligned_pv(int(ncid))
        if s is not None and not s.empty:
            fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines',
                                     line=dict(color='lightgray', width=1),
                                     showlegend=False, hoverinfo='skip'), row=1, col=col)
    sq = aligned_pv(qc)
    if sq is not None and not sq.empty:
        fig.add_trace(go.Scatter(x=sq.index, y=sq.values, mode='lines',
                                 line=dict(color='red', width=2.5),
                                 showlegend=False), row=1, col=col)
    fig.add_hline(y=ALARM_THRESHOLD, line_dash='dash', line_color='red', opacity=0.5, row=1, col=col)
    fig.add_vline(x=0, line_dash='dot', line_color='blue', opacity=0.3, row=1, col=col)

fig.update_yaxes(range=[15, 50], title_text='03LIC_1071.PV', row=1, col=1)
fig.update_xaxes(title_text='min from trough')
fig.update_layout(height=400, width=1400,
                  title_text='Nearest-neighbour validation (red = query, gray = its 5 descriptor-space neighbours)')
fig.show()
fig.write_html(os.path.join(OUT_DIR, 'similarity_neighbours.html'))
print(f'Saved -> {OUT_DIR}/similarity_neighbours.html')


Saved -> /home/h604827/ControlActions/RESULTS/episode_trajectory_analysis/similarity_neighbours.html


In [24]:
# Export a small index of representative examples per archetype (for SME review / plotting)
examples = []
for arch in char_order:
    sub = desc[desc['character'] == arch].copy()
    if sub.empty:
        continue
    sub = sub.sort_values('depth_below_limit')
    picks = pd.concat([sub.head(2), sub.iloc[[len(sub) // 2]], sub.tail(2)]).drop_duplicates('cluster_id')
    for _, r in picks.iterrows():
        examples.append({
            'character': arch, 'severity': r['severity'], 'cluster_id': int(r['cluster_id']),
            'onset': r['onset'], 'depth_below_limit': round(r['depth_below_limit'], 2),
            'alarm_duration_min': r['alarm_duration_min'],
            'approach_slope_30': round(r['approach_slope_30'], 3),
            'n_troughs_near_limit': int(r['n_troughs_near_limit']),
            'sme_retained': bool(r['sme_retained']),
            'plot_html': os.path.join('all_episodes', f'episode_{int(r["cluster_id"]):04d}',
                                      f'episode_{int(r["cluster_id"]):04d}_plot.html'),
        })
examples_df = pd.DataFrame(examples)
examples_df.to_csv(os.path.join(OUT_DIR, 'archetype_example_index.csv'), index=False)
print(f'Saved {len(examples_df)} representative examples -> {OUT_DIR}/archetype_example_index.csv')
examples_df


Saved 25 representative examples -> /home/h604827/ControlActions/RESULTS/episode_trajectory_analysis/archetype_example_index.csv


,character,severity,cluster_id,onset,depth_below_limit,alarm_duration_min,approach_slope_30,n_troughs_near_limit,sme_retained,plot_html
0,trip,deep,105,2022-10-30 17:42:46.502,26.56,109.703333,-0.010,6,False,all_episodes/episode_0105/episode_0105_plot.html
1,trip,deep,1,2022-01-05 08:53:41.853,28.68,39.854200,-0.907,3,False,all_episodes/episode_0001/episode_0001_plot.html
2,trip,deep,214,2023-12-26 17:53:23.202,29.29,145.165033,-0.744,7,False,all_episodes/episode_0214/episode_0214_plot.html
3,trip,deep,534,2025-06-21 15:47:15.387,30.08,281.314283,-0.152,39,False,all_episodes/episode_0534/episode_0534_plot.html
4,trip,deep,518,2025-05-22 16:33:43.210,30.09,178.549350,-0.445,25,False,all_episodes/episode_0518/episode_0518_plot.html
5,graze,shallow,55,2022-04-02 17:22:34.252,0.00,3.617533,-0.160,1,False,all_episodes/episode_0055/episode_0055_plot.html
6,graze,shallow,86,2022-08-05 13:18:38.201,0.00,0.950017,-0.007,0,False,all_episodes/episode_0086/episode_0086_plot.html
7,graze,shallow,377,2024-06-24 10:55:24.356,0.23,2.379117,-0.178,1,True,all_episodes/episode_0377/episode_0377_plot.html
8,graze,shallow,265,2024-01-28 00:04:06.853,0.97,4.066683,-0.191,4,False,all_episodes/episode_0265/episode_0265_plot.html
9,graze,shallow,310,2024-05-09 08:48:33.753,0.98,4.291667,-0.090,1,True,all_episodes/episode_0310/episode_0310_plot.html


## 6. How do we know episodes in a category are *actually* similar?

This is the right question to push on. The visual overlays in §4.2 and §5 are suggestive but subjective, so here are **two quantitative checks, neither of which trusts the labels**:

1. **Feature space** — do episodes sit close to other same-label episodes?
   - **Silhouette** (−1…+1): +1 = tight and well separated, ~0 = on a boundary, <0 = sits closer to another group (likely mislabelled).
   - **Leave-one-out nearest-neighbour agreement**: for each episode, find its single closest episode and check whether it has the same label. High % = the categories are locally consistent.
   - **Within vs between**: average distance to your own group's centre vs the nearest *other* group's centre (<1 ratio = tighter to your own group).
2. **Shape space** (independent of the features) — align the **raw** PV at the trough and check each curve against its category's *average shape*.

**Honest caveat stated up front:** the categories are **rule-based**, so they are similar *by construction* on the 4 rule features (duration, depth, #troughs, frac-below). The real value of these checks is whether episodes in a category are **also** similar on the features that did **not** go into the rules (the descent/recovery slopes, volatility) and on raw trajectory shape. If yes, the categories are capturing something real, not just echoing the thresholds.


In [27]:
# How coherent are the categories? Two independent checks that do NOT trust the labels.
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.spatial.distance import cdist

coh = desc[desc['character'] != 'trip'].copy().reset_index(drop=True)

# Standardise the SAME 7 features used by the similarity metric (robust median/IQR)
Xc = coh[sim_feats].astype(float)
medc = Xc.median()
iqrc = (Xc.quantile(0.75) - Xc.quantile(0.25)).replace(0, 1.0)
Z = ((Xc - medc) / iqrc).fillna(0).values
labels = coh['character'].values

# ---- (1) FEATURE SPACE ----
# Silhouette: +1 = tight & well separated, 0 = on the boundary, <0 = likely mislabelled
coh['silhouette'] = silhouette_samples(Z, labels)
print(f"(1a) Silhouette (feature space): overall = {silhouette_score(Z, labels):.3f}")
print(coh.groupby('character')['silhouette'].mean().reindex(
    [c for c in char_order if c != 'trip']).round(3).to_string())

# Leave-one-out nearest neighbour: does each episode's single closest neighbour share its label?
Dm = cdist(Z, Z); np.fill_diagonal(Dm, np.inf)
coh['nn_same_label'] = labels == labels[Dm.argmin(axis=1)]
print(f"\n(1b) Leave-one-out NN label agreement: overall = {coh['nn_same_label'].mean()*100:.0f}%")
print((coh.groupby('character')['nn_same_label'].mean() * 100).reindex(
    [c for c in char_order if c != 'trip']).round(0).to_string())

# Within vs between: distance to own centroid vs nearest other-category centroid
cents = {c: Z[labels == c].mean(axis=0) for c in np.unique(labels)}
own = np.array([np.linalg.norm(Z[i] - cents[labels[i]]) for i in range(len(Z))])
oth = np.array([min(np.linalg.norm(Z[i] - cents[c]) for c in cents if c != labels[i])
                for i in range(len(Z))])
print(f"\n(1c) Mean distance to OWN centroid = {own.mean():.2f}  vs  nearest OTHER = {oth.mean():.2f}  "
      f"(ratio {own.mean()/oth.mean():.2f}; <1 means tighter to own group)")

# ---- (2) SHAPE SPACE (independent of the features) ----
# Align raw PV at the trough, z-normalise each curve, compare to each category's average shape.
GRID = np.arange(-60, 61)
curves = {}
for cid in coh['cluster_id']:
    s = aligned_pv(cid, lo=-60, hi=60)
    if s is None or s.empty:
        continue
    s = s.reindex(GRID).interpolate(limit_direction='both')
    sd = s.std()
    curves[cid] = ((s - s.mean()) / (sd if sd and sd > 0 else 1.0)).values
cur_df = pd.DataFrame(curves).T
lbl_by_cid = coh.set_index('cluster_id')['character']
cat_mean = {c: cur_df.loc[[i for i in cur_df.index if lbl_by_cid[i] == c]].mean().values
            for c in np.unique(labels)}
best_shape = {}
for cid in cur_df.index:
    v = cur_df.loc[cid].values
    best_shape[cid] = min(cat_mean, key=lambda c: np.sqrt(np.nanmean((v - cat_mean[c]) ** 2)))
coh['shape_best'] = coh['cluster_id'].map(best_shape)
coh['shape_match'] = coh['shape_best'] == coh['character']
print(f"\n(2) Shape purity (raw-curve match to own category's mean shape): "
      f"overall = {coh['shape_match'].mean()*100:.0f}%")
print((coh.groupby('character')['shape_match'].mean() * 100).reindex(
    [c for c in char_order if c != 'trip']).round(0).to_string())
print("\nNote: grazes are defined by brevity/shallowness, not a distinctive multi-point shape,")
print("so their shape-purity is naturally lower — feature space is the right lens for them.")


(1a) Silhouette (feature space): overall = -0.035
character
graze            0.281
oscillatory     -0.092
transient_dip   -0.102
sustained_dip   -0.164

(1b) Leave-one-out NN label agreement: overall = 74%
character
graze            64.0
oscillatory      72.0
transient_dip    80.0
sustained_dip    73.0

(1c) Mean distance to OWN centroid = 2.23  vs  nearest OTHER = 2.39  (ratio 0.93; <1 means tighter to own group)

(2) Shape purity (raw-curve match to own category's mean shape): overall = 39%
character
graze            45.0
oscillatory      41.0
transient_dip    31.0
sustained_dip    57.0

Note: grazes are defined by brevity/shallowness, not a distinctive multi-point shape,
so their shape-purity is naturally lower — feature space is the right lens for them.


## 7. Episode → category assignments (which episode is in which category)

The full per-episode listing, sorted by `character` → `severity` → depth. Saved to `RESULTS/episode_trajectory_analysis/episode_category_assignments.csv`. The helper `episodes_in('oscillatory')` (or with a severity, e.g. `episodes_in('sustained_dip', 'deep')`) returns the episodes in any category so you can pull up their plots in `all_episodes/episode_XXXX/episode_XXXX_plot.html`.


In [ ]:
# Per-episode category assignment table (full listing) + CSV
assign = desc[['cluster_id', 'onset', 'character', 'severity',
               'depth_below_limit', 'alarm_duration_min', 'max_descent_rate',
               'approach_slope_30', 'recovery_slope_30',
               'n_troughs_near_limit', 'frac_below_limit',
               'is_trip', 'sme_retained']].copy()
assign = assign.round({'depth_below_limit': 2, 'alarm_duration_min': 1,
                       'max_descent_rate': 3, 'approach_slope_30': 3,
                       'recovery_slope_30': 3, 'frac_below_limit': 3})
assign = assign.sort_values(['character', 'severity', 'depth_below_limit']).reset_index(drop=True)

out_csv = os.path.join(OUT_DIR, 'episode_category_assignments.csv')
assign.to_csv(out_csv, index=False)

print('Counts by character:')
print(assign['character'].value_counts().reindex(char_order).to_string())
print('\nCharacter x severity:')
print(pd.crosstab(assign['character'], assign['severity']).reindex(char_order)[['shallow', 'moderate', 'deep']].to_string())
print(f'\nSaved full assignment -> {out_csv}')


def episodes_in(character=None, severity=None):
    """Convenience: list cluster_ids in a given category (and optional severity)."""
    m = pd.Series(True, index=assign.index)
    if character is not None:
        m &= assign['character'].eq(character)
    if severity is not None:
        m &= assign['severity'].eq(severity)
    cols = ['cluster_id', 'onset', 'depth_below_limit', 'alarm_duration_min',
            'n_troughs_near_limit', 'frac_below_limit', 'sme_retained']
    return assign.loc[m, cols].reset_index(drop=True)


print(f"\nExample: episodes_in('oscillatory') -> {len(episodes_in('oscillatory'))} episodes; "
      f"episodes_in('sustained_dip','deep') -> {len(episodes_in('sustained_dip','deep'))} episodes")

# Full table (scroll to browse; trips are at the top, then graze -> sustained_dip)
assign


## 8. Findings, opinion, and proposed next steps

> **Decision (agreed):** keep all episodes, **drop trips only**. The SME funnel is kept purely as a `filter_outcome` column for reference / optional feed-cause labelling — it is not used to shrink the dataset.

> **Data-correctness note (this revision).** The alarm window is now taken from the **events-based cluster boundaries** (`cluster_start_time` / `cluster_end_time`), not the PV-file `AlarmStatus` column. Switching mattered: 4 episodes had no `ON` samples at all (now recovered → all **539** described), and because `AlarmStatus` often turned `ON` far back in the pre-context, the **median alarm duration dropped from ~52 min to ~7 min** and the trough `pv_min` changed in ~23% of episodes. The numbers below reflect the corrected windows.

### What we learned about the alarm population
- The 539 PVLO episodes are **highly heterogeneous**. The descriptor splits the non-trip population (**519**) into: **graze 97**, **oscillatory 114**, **transient_dip 248**, **sustained_dip 60**, plus **20 trips** (PV → 0, shutdowns).
- Your observation is confirmed and quantified: a large share of alarms are **not clean dips**. `graze` + `oscillatory` = **211 / 519 (41%)** are momentary touches or ongoing oscillation that merely brushes the limit. The baseline PV is volatile almost everywhere (median pre-alarm σ ≈ 3.4).
- With the corrected (tighter) windows, **genuinely sustained excursions are rarer than they looked** — only 60 episodes (~12%) truly stay below the limit; the bulk are short transient dips.
- §6 quantifies that the categories are coherent (silhouette / nearest-neighbour agreement), and §7 lists exactly which episode is in which category.

### On the SME filtering (kept as a label, not a filter)
- The funnel keeps **33% of grazes** but only **6.7% of sustained_dips** (and **14%** of oscillatory). It is biased *toward* the easy/short alarms and *against* the long, genuine excursions where control action matters most.
- The reason is **Stage 2 (FI1000 in-band)** — effectively a **feed-cause selector, not a quality filter**. We therefore keep it only as an optional cause label and rely on *drop-trips-only* for the working set.

### Bridge to the OTS test (what the seniors want to see)
The OTS test = run under 3 feed scenarios of increasing severity and check that **suggested action magnitude scales with severity** (and points the right way). This notebook gives the severity signals that make that possible and defensible:
- `depth_below_limit` (how far below the limit) and `max_descent_rate` / `approach_slope_30` (how fast it's falling).
- The 3 feed scenarios (500 → 480 → 460 T/D) will produce **monotonically different descent rates / depths**, which a recommender can map to **monotonically different action magnitudes**.

### Proposed next steps (for your decision)
1. **Cause labelling** — split the non-trip episodes into *feed-driven* vs *pressure-driven (03PIC_1013)* vs *other*, using the feed/pressure context features already in the descriptor (+ FI1000 band).
2. **Action coupling** — join each episode to the control actions actually taken (from the clustered workbook) and check, per archetype/cause, the **direction + magnitude** of operator OP/SP steps vs the severity features. This is what turns the descriptor into an action recommender.
3. **Severity → magnitude curve** — fit a simple, monotonic relationship (severity feature → operator step size) so the OTS scenarios produce scaling magnitudes.

> **Remaining questions for you:**
> (b) Should I prioritise **cause labelling** (step 1) or jump straight to **action coupling** (step 2) next?
> (c) Are these archetype thresholds reasonable, or do you want to tune any (e.g. what counts as a "graze")? §3.1 also flags whether you'd like oscillation measured only in the alarm-local window.


In [72]:
import pandas as pd

df = pd.read_excel('/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx', sheet_name='control_actions')
# Safe numeric step calculation (non-numeric values become NaN)
df['Step'] = (
    pd.to_numeric(df['Value'], errors='coerce')
    - pd.to_numeric(df['PrevValue'], errors='coerce')
)

In [73]:
df['filtered'].value_counts()

filtered
False    38446
True      3844
Name: count, dtype: int64

In [74]:
df = df[
    (df['Source'] == '03HIC_1141') &
    # (df['filtered'] == True) & 
    # (df['Description'] == 'OP') & 
    # (df['action_timing'] == 'during') #&
    (pd.to_datetime(df['VT_Start']) <= pd.to_datetime(df['cluster_start'])) &
    (pd.to_datetime(df['VT_Start']) >= pd.to_datetime(df['cluster_start']) - pd.Timedelta(minutes=30)) #& 
    # (df['cluster_id'].isin([532, 332, 377, 363, 199, 493, 382, 244, 300]))
]
df['action_direction'].value_counts()

action_direction
decrease     258
increase     147
no_change     21
Name: count, dtype: int64

In [75]:
df['cluster_id'].value_counts()

cluster_id
177    51
1      49
392    32
50     31
77     20
214    17
504    16
287    15
87     14
463    13
54     12
46     12
90     12
55     10
121     8
73      8
51      7
503     7
213     6
136     6
78      6
215     6
370     5
56      5
59      5
152     5
89      4
500     4
57      3
184     3
178     3
127     3
511     3
49      3
402     2
120     2
91      2
93      2
74      2
76      2
17      1
14      1
75      1
62      1
58      1
61      1
151     1
270     1
289     1
438     1
Name: count, dtype: int64

In [68]:
df[df['action_direction'] == 'increase']['Step'].describe()

count    147.000000
mean       2.071339
std        1.934626
min        0.099900
25%        2.000000
50%        2.000000
75%        2.000000
max       20.000000
Name: Step, dtype: float64

In [69]:
df[df['action_direction'] == 'decrease']['Step'].describe()

count    258.000000
mean      -1.064923
std        0.901544
min       -3.000000
25%       -2.000000
50%       -1.000000
75%       -0.100000
max       -0.100000
Name: Step, dtype: float64